In [1]:
import os
os.environ["DISABLE_PANDERA_IMPORT_WARNING"] = "True"

from network_wrangler.transit import write_transit

In [1]:
import os
import sys
import numpy as np

from pyproj import CRS
from pathlib import Path

from projectcard import read_cards

import network_wrangler
from network_wrangler import load_roadway
from network_wrangler import load_transit
from network_wrangler import create_scenario
from network_wrangler import Scenario
from network_wrangler.roadway import write_roadway
from network_wrangler.transit import write_transit

from met_council_wrangler import MetCouncil_Parameters
from met_council_wrangler import metcouncil_roadway
from met_council_wrangler import metcouncil_transit

from cube_wrangler import Parameters
from cube_wrangler import util
from cube_wrangler import roadway
from cube_wrangler import StandardTransit


c:\Users\USLP095001\code\met_council\met_council_wrangler\.venv\lib\site-packages\network_wrangler\models\roadway\tables.py:41: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  class RoadLinksTable(DataFrameModel):


In [2]:
from pathlib import Path
input_dir = Path(r"C:\Users\USLP095001\WSP O365\Met Council Network Wrangler Updates (26P088) - Documents\General\Task 1. Base Network Improvements\1.1 Centroid Connectors\inputs")
output_dir = Path(r"C:\Users\USLP095001\WSP O365\Met Council Network Wrangler Updates (26P088) - Documents\General\Task 1. Base Network Improvements\1.1 Centroid Connectors")


In [3]:
current_network = load_roadway(
    input_dir / "links.json",
    input_dir / "nodes.geojson",
    input_dir / "shapes.geojson",
)

In [4]:
import pandas as pd
centroid_nodes = pd.read_pickle(input_dir / "centroid_node.pickle")
cc_links = pd.read_pickle(input_dir / "cc_link.pickle")
cc_shapes = pd.read_pickle(input_dir / "cc_shape.pickle")

C:\Users\USLP095001\AppData\Local\miniforge3\lib\pickle.py:1718: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  setstate(state)
C:\Users\USLP095001\AppData\Local\miniforge3\lib\pickle.py:1718: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  setstate(state)
C:\Users\USLP095001\AppData\Local\miniforge3\lib\pickle.py:1718: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  setstate(state)


In [5]:
import numpy as np
import pandas as pd
import geopandas as gpd

from network_wrangler.roadway.nodes.create import data_to_nodes_df
from network_wrangler.roadway.shapes.create import df_to_shapes_df
from network_wrangler.roadway.links.create import data_to_links_df

# Work only on copies so the original inputs remain unchanged.
centroid_nodes_work = centroid_nodes.copy()
cc_shapes_work = cc_shapes.copy()
cc_links_work = cc_links.copy()

# Convert nodes to Wrangler format.
centroid_nodes_work["drive_access"] = True
centroid_nodes_work["walk_access"] = True
centroid_nodes_work["bike_access"] = True
centroid_nodes_updated = data_to_nodes_df(
    centroid_nodes_work.drop(columns=["OBJECTID", "N", "model_node"]),
    config=current_network.config,
)

# Convert shapes to Wrangler format.
if "shape_id" not in cc_shapes_work.columns and "id" in cc_shapes_work.columns:
    cc_shapes_work = cc_shapes_work.rename(columns={"id": "shape_id"})

cc_shapes_updated = df_to_shapes_df(
    cc_shapes_work,
    config=current_network.config,
)

# Add required connector-link attributes.
first_link_id = int(current_network.links_df["model_link_id"].max()) + 1
cc_links_work["model_link_id"] = np.arange(
    first_link_id,
    first_link_id + len(cc_links_work),
)
cc_links_work["lanes"] = 1
cc_links_work["roadClass"] = "Centroid"
cc_links_work["oneWay"] = False

for column in ["drive_access", "walk_access", "bike_access"]:
    if column in cc_links_work.columns:
        cc_links_work[column] = cc_links_work[column].isin([True, 1, "1"])

# Align node CRS before concatenating.
current_nodes_for_conversion = current_network.nodes_df.to_crs(
    centroid_nodes_updated.crs
)

all_nodes = gpd.GeoDataFrame(
    pd.concat(
        [current_nodes_for_conversion, centroid_nodes_updated],
        ignore_index=True,
    ),
    geometry="geometry",
    crs=centroid_nodes_updated.crs,
)

# Convert links to Wrangler format.
cc_links_updated = data_to_links_df(
    cc_links_work,
    nodes_df=all_nodes,
)

# Match each output's CRS to the corresponding current-network table.
centroid_nodes_updated = centroid_nodes_updated.to_crs(
    current_network.nodes_df.crs
)
cc_shapes_updated = cc_shapes_updated.to_crs(
    current_network.shapes_df.crs
)
cc_links_updated = cc_links_updated.to_crs(
    current_network.links_df.crs
)

In [18]:
current_network.add_nodes(centroid_nodes_updated)
current_network.add_links(cc_links_updated)
current_network.add_shapes(cc_shapes_updated)

In [21]:
write_roadway(
    current_network,
    output_dir
)

In [22]:
current_network.shapes_df.to_parquet(output_dir / "raw_network.parquet")

In [23]:
from keplergl import KeplerGl

# Kepler.gl expects WGS84 coordinates.
all_shapes = current_network.shapes_df.to_crs(epsg=4326).copy()
centroid_points = centroid_nodes_updated.to_crs(epsg=4326).copy()

# Separate centroid connectors from the rest of the roadway shapes.
centroid_shape_ids = set(cc_shapes_updated["shape_id"].astype(str))
is_centroid_connector = all_shapes["shape_id"].astype(str).isin(centroid_shape_ids)

roadway_shapes = all_shapes.loc[~is_centroid_connector].copy()
centroid_connectors = all_shapes.loc[is_centroid_connector].copy()

# Center the initial view on the centroids.
min_x, min_y, max_x, max_y = centroid_points.total_bounds
center_lon = (min_x + max_x) / 2
center_lat = (min_y + max_y) / 2

config = {
    "version": "v1",
    "config": {
        "visState": {
            "layers": [
                {
                    "id": "roadway-shapes",
                    "type": "geojson",
                    "config": {
                        "dataId": "Roadway network",
                        "label": "Roadway network",
                        "color": [120, 120, 120],
                        "isVisible": True,
                        "visConfig": {
                            "opacity": 0.35,
                            "strokeOpacity": 0.35,
                            "thickness": 0.4,
                            "strokeColor": [120, 120, 120],
                        },
                    },
                },
                {
                    "id": "centroid-connectors",
                    "type": "geojson",
                    "config": {
                        "dataId": "Centroid connectors",
                        "label": "Centroid connectors",
                        "color": [255, 140, 0],
                        "isVisible": True,
                        "visConfig": {
                            "opacity": 0.9,
                            "strokeOpacity": 0.9,
                            "thickness": 2,
                            "strokeColor": [255, 140, 0],
                        },
                    },
                },
                {
                    "id": "centroid-points",
                    "type": "geojson",
                    "config": {
                        "dataId": "Centroids",
                        "label": "Centroids",
                        "color": [220, 20, 60],
                        "isVisible": True,
                        "visConfig": {
                            "opacity": 0.9,
                            "radius": 5,
                            "filled": True,
                            "color": [220, 20, 60],
                            "strokeColor": [255, 255, 255],
                            "strokeWidth": 1,
                        },
                    },
                },
            ]
        },
        "mapState": {
            "latitude": center_lat,
            "longitude": center_lon,
            "zoom": 8,
        },
    },
}

kepler_map = KeplerGl(
    height=750,
    data={
        "Roadway network": roadway_shapes,
        "Centroid connectors": centroid_connectors,
        "Centroids": centroid_points,
    },
    config=config,
)

html_output = output_dir / "centroid_network_map.html"

kepler_map.save_to_html(
    file_name=str(html_output),
    read_only=False,
)

print(f"Saved interactive map to: {html_output}")

Map saved to C:\Users\USLP095001\WSP O365\Met Council Network Wrangler Updates (26P088) - Documents\General\Task 1. Base Network Improvements\1.1 Centroid Connectors\centroid_network_map.html!
Saved interactive map to: C:\Users\USLP095001\WSP O365\Met Council Network Wrangler Updates (26P088) - Documents\General\Task 1. Base Network Improvements\1.1 Centroid Connectors\centroid_network_map.html
